# JAX CFD 系统课程 · Lesson 02
## 对流方程、数值耗散、色散与 CFL 稳定性

**课程路线：** 物理问题 → 数学推导 → LaTeX → 离散格式 → JAX 实现 → 数值验证 → 二维推广。

在上一课，我们实现了 $q_t=\nu\nabla^2q$（扩散）；本课开始求解 **输运**：

$$
\boxed{\frac{\partial q}{\partial t}+c\frac{\partial q}{\partial x}=0.}
$$

完成后，你应该能回答：

1. 为什么一阶迎风格式会让波形变钝？
2. 为什么中心差分的空间精度更高，配合显式欧拉却不稳定？
3. 为什么二阶 Lax–Wendroff 格式在间断附近可能振荡？
4. CFL 数为什么决定显式格式的稳定范围？
5. 如何把一维对流求解器推广到二维 JAX CFD 程序？

> 目标是构建**教学型 CFD 组件**，不是直接替代你毕设中的高保真圆柱绕流求解器。

### 环境与 Notebook 操作

在项目根目录执行（已经装过依赖时可以跳过）：

```bash
uv add jax numpy matplotlib scipy
uv add --dev ipykernel
```

VS Code 安装 Microsoft **Python** 与 **Jupyter** 扩展；打开 `.ipynb`，右上角选择项目 `.venv` 的 Python 解释器。

- `Shift + Enter`：运行当前单元格并移动到下一格
- `Ctrl/Cmd + Enter`：运行当前单元格并停留
- Markdown 单元格使用 `$...$`（行内）和 `$$...$$`（独立公式）
- 修改了函数定义后要重新运行定义单元格；调试时用 **Restart Kernel → Run All** 检查执行顺序
- 第一次 JIT 调用包含编译时间；计时要先预热，并 `.block_until_ready()`

**完整 Python 文件**也可以放在同目录运行，但建议这次逐个执行单元格，自己观察图像与误差。

## 1. 物理模型和解析解

一维匀速对流方程：

$$
\frac{\partial q}{\partial t}+c\frac{\partial q}{\partial x}=0,
\qquad c=\text{常数}.
$$

沿特征线 $x-ct=\text{常数}$，场值保持不变：

$$
\boxed{q(x,t)=q_0(x-ct).}
$$

采用周期边界 $q(0,t)=q(L,t)$，相当于波形从右边离开后从左边重新进入。

今天选取长度为 $L=1$ 的周期区间，光滑初值是一段窄的周期高斯脉冲；因此可以构造解析参考解，并在经过边界时检验周期处理是否正确。

**LaTeX 写法：**

```latex
$$
\frac{\partial q}{\partial t}
+c\frac{\partial q}{\partial x}=0
$$

$$
q(x,t)=q_0(x-ct)
$$
```

> 注意本课的 $q$ 是**被输运的标量**，$c$ 是流速。最终 Navier–Stokes 中，速度场本身也参与对流，而且速度一般不恒定。

In [ ]:
import math
import time
from functools import partial

import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from jax import lax

# 先使用 CPU 上适合数值实验的双精度，减少舍入误差干扰。
jax.config.update("jax_enable_x64", True)

print("JAX version:", jax.__version__)
print("Available devices:", jax.devices())

## 2. 网格、周期边界与 JAX 数组

$N$ 个网格点位于：

$$
x_j=j\Delta x,\qquad \Delta x=\frac{L}{N},\qquad j=0,1,\dots,N-1.
$$

与上一课的 Dirichlet 边界不同，**周期网格不重复存储右端点** $x=L$，因为它与 $x=0$ 是同一个物理点。代码中使用 `endpoint=False`。

周期边界下：

$$
q_{-1}=q_{N-1},\qquad q_N=q_0.
$$

这正好可以用 `jnp.roll` 实现；`jnp.roll(q,1)` 在位置 $j$ 存放原来位置 $j-1$ 的值，`jnp.roll(q,-1)` 则相反。

In [ ]:
a = jnp.array([10, 20, 30, 40])
print("Original :", np.asarray(a))
print("Left nbr:", np.asarray(jnp.roll(a, 1)))
print("Right nbr:", np.asarray(jnp.roll(a, -1)))
assert np.array_equal(np.asarray(jnp.roll(a, 1)), np.array([40, 10, 20, 30]))

### 2.1 初始化周期高斯脉冲

定义周期最短距离

$$
d_L(x,x_0)=\left((x-x_0+L/2)\bmod L\right)-L/2.
$$

则

$$
q_0(x)=\exp\left[-\frac{d_L(x,x_0)^2}{2\sigma^2}\right].
$$

这样脉冲穿过周期边界也能保持正确形状。初始中心 $x_0=0.30$，宽度 $\sigma=0.065$。

In [ ]:
L = 1.0
N = 256
c = 1.0                  # 本课先研究 c > 0
x0 = 0.30
sigma = 0.065

dx = L / N
x = jnp.linspace(0.0, L, N, endpoint=False)

def periodic_distance(x, center, length=1.0):
    return jnp.mod(x - center + length / 2, length) - length / 2

def gaussian(x, center, sigma, length=1.0):
    distance = periodic_distance(x, center, length)
    return jnp.exp(-0.5 * (distance / sigma) ** 2)

q0 = gaussian(x, x0, sigma, L)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(np.asarray(x), np.asarray(q0), lw=2)
ax.set(xlabel="x", ylabel="q(x,0)", title="Initial smooth periodic pulse")
ax.grid(alpha=0.25)
plt.show()

## 3. 空间差分与迎风格式

假设 $c>0$，信息从左向右传播。用**左侧**节点构造一阶迎风导数：

$$
\left.\frac{\partial q}{\partial x}\right|_j
\approx\frac{q_j-q_{j-1}}{\Delta x}.
$$

时间方向用显式欧拉：

$$
\frac{q_j^{n+1}-q_j^n}{\Delta t}
+c\frac{q_j^n-q_{j-1}^n}{\Delta x}=0.
$$

于是

$$
\boxed{q_j^{n+1}=q_j^n-C(q_j^n-q_{j-1}^n),\quad C=\frac{c\Delta t}{\Delta x}.}
$$

$C$ 是 Courant 数。对这个**正速度 + 一阶迎风 + 显式欧拉**组合，其稳定范围为 $0\le C\le 1$。

**LaTeX 多行对齐：**

```latex
$$
\begin{aligned}
q_j^{n+1}
&=q_j^n-c\Delta t\frac{q_j^n-q_{j-1}^n}{\Delta x}\\
&=q_j^n-C(q_j^n-q_{j-1}^n).
\end{aligned}
$$
```

提示：`&` 是对齐锚点，`\\` 是换行，`\Delta` 是大写希腊 Delta。

## 4. 与迎风比较：中心差分和 Lax–Wendroff

**中心差分 + 显式欧拉：**

$$
q_j^{n+1}=q_j^n-\frac{C}{2}(q_{j+1}^n-q_{j-1}^n).
$$

空间上为二阶，但这一时间-空间组合对纯对流不稳定，不能简单地认为“二阶就更好”。

**Lax–Wendroff（二阶单步）：**

$$
\boxed{
q_j^{n+1}=q_j^n
-\frac{C}{2}(q_{j+1}^n-q_{j-1}^n)
+\frac{C^2}{2}(q_{j+1}^n-2q_j^n+q_{j-1}^n).
}
$$

Lax–Wendroff 在光滑解上具有二阶精度，$|C|\le1$ 时满足线性稳定性条件；但对不连续的初始场，可能出现上下过冲（数值色散 / 振荡）。

In [ ]:
def step_upwind(q, C):
    """c > 0, 1D periodic upwind + forward Euler."""
    left = jnp.roll(q, 1)
    return q - C * (q - left)

def step_central(q, C):
    """Central spatial derivative + forward Euler (unstable for advection)."""
    left = jnp.roll(q, 1)
    right = jnp.roll(q, -1)
    return q - 0.5 * C * (right - left)

def step_lax_wendroff(q, C):
    """Second-order Lax–Wendroff for constant-speed advection."""
    left = jnp.roll(q, 1)
    right = jnp.roll(q, -1)
    return q - 0.5 * C * (right - left) + 0.5 * C**2 * (right - 2 * q + left)

# 小型单步检查：周期网格下，各格式都会守恒离散和。
test = jnp.array([0.2, 0.5, 0.7, 0.1])
for name, f in [('upwind', step_upwind), ('central', step_central), ('LW', step_lax_wendroff)]:
    result = f(test, 0.6)
    assert np.isclose(float(jnp.sum(test)), float(jnp.sum(result)))
    print(f"{name:14s}: {np.asarray(result)}")

## 5. 使用 JAX 编译多个时间步

上节课你接触过 `jax.jit` 与 `lax.fori_loop`。现在我们将不同数值格式封装到同一个求解接口中。

在 JAX 中，`scheme` 和 `nsteps` 被标记为**静态参数**，这样编译时可以决定使用哪个离散公式、迭代多少次。不同格式或步数可能触发重新编译。

对于周期均匀网格，质量（积分）近似为：

$$
M(t)=\int_0^L q(x,t)\,\mathrm{d}x
\approx \Delta x\sum_{j=0}^{N-1}q_j(t).
$$

周期线性对流在理论上守恒总质量。我们会把这条守恒规律当作一个程序检验。

In [ ]:
@partial(jax.jit, static_argnames=("nsteps", "scheme"))
def evolve_1d(q_initial, C, nsteps, scheme="upwind"):
    if scheme == "upwind":
        step_fn = step_upwind
    elif scheme == "central":
        step_fn = step_central
    elif scheme == "lax_wendroff":
        step_fn = step_lax_wendroff
    else:
        raise ValueError(f"Unknown scheme: {scheme}")
    return lax.fori_loop(0, nsteps, lambda _, q: step_fn(q, C), q_initial)

def step_count_for_target(T, dx, speed, target_cfl=0.65):
    nsteps = max(1, math.ceil(T * abs(speed) / (target_cfl * dx)))
    dt = T / nsteps
    return nsteps, dt, abs(speed) * dt / dx

## 6. 实验 A：光滑高斯脉冲的平移

设置终止时间 $T=0.80$，脉冲中心由 $x_0=0.30$ 移到 $x_0+cT=1.10$，因此它会**穿过周期边界**并出现在 $x\approx0.10$。

我们对比一阶迎风与 Lax–Wendroff，并用解析解作为参考。

离散 $L^1$ 误差：

$$
E_1=\Delta x\sum_{j=0}^{N-1}|q_j^{\mathrm{num}}-q_j^{\mathrm{exact}}|.
$$

离散 $L^2$ 误差：

$$
E_2=\sqrt{\Delta x\sum_{j=0}^{N-1}(q_j^{\mathrm{num}}-q_j^{\mathrm{exact}})^2}.
$$

In [ ]:
T = 0.80
nsteps, dt, C = step_count_for_target(T, dx, c, target_cfl=0.65)
q_exact = gaussian(x, x0 + c*T, sigma, L)
q_upwind = evolve_1d(q0, C, nsteps, "upwind")
q_lw = evolve_1d(q0, C, nsteps, "lax_wendroff")

print(f"N={N}, dx={dx:.6f}, T={T}, nsteps={nsteps}, dt={dt:.6f}, C={C:.4f}")
assert 0 < C <= 1
assert np.all(np.isfinite(np.asarray(q_upwind)))
assert np.all(np.isfinite(np.asarray(q_lw)))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(np.asarray(x), np.asarray(q_exact), 'k--', lw=2, label='Exact')
ax.plot(np.asarray(x), np.asarray(q_upwind), lw=1.6, label='Upwind + Euler')
ax.plot(np.asarray(x), np.asarray(q_lw), lw=1.6, label='Lax–Wendroff')
ax.set(xlabel='x', ylabel='q(x,T)', title='Smooth pulse: translation through periodic boundary')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

In [ ]:
def diagnostics(q_num, q_ref, q_start, dx):
    diff = q_num - q_ref
    return {
        'L1_error': float(jnp.sum(jnp.abs(diff)) * dx),
        'L2_error': float(jnp.sqrt(jnp.sum(diff**2) * dx)),
        'Linf_error': float(jnp.max(jnp.abs(diff))),
        'mass_change': float((jnp.sum(q_num) - jnp.sum(q_start)) * dx),
        'min': float(jnp.min(q_num)),
        'max': float(jnp.max(q_num)),
    }

for name, result in [('Upwind', q_upwind), ('Lax–Wendroff', q_lw)]:
    d = diagnostics(result, q_exact, q0, dx)
    print(f"{name:15s} L1={d['L1_error']:.6e}  L2={d['L2_error']:.6e}  "
          f"mass_change={d['mass_change']:+.2e}  max={d['max']:.6f}")

assert abs(diagnostics(q_upwind, q_exact, q0, dx)['mass_change']) < 1e-10
assert abs(diagnostics(q_lw, q_exact, q0, dx)['mass_change']) < 1e-10

### 为什么迎风格式会使高斯峰变矮、变宽？

对于 $c>0$，迎风 + 显式欧拉的**修正方程**首要附加项为：

$$
\frac{\partial q}{\partial t}
+c\frac{\partial q}{\partial x}
=\underbrace{\frac{c\Delta x}{2}(1-C)}_{\nu_{\mathrm{num}}}
\frac{\partial^2q}{\partial x^2}+\text{高阶项}.
$$

也就是说，离散误差可以在主导阶上等效为一个**人为扩散项**，扩散系数 $\nu_{\mathrm{num}}=c\Delta x(1-C)/2$。

这并非真实流体的黏性，而是数值格式引入的**数值耗散**。如果将来模拟圆柱涡街时数值耗散过大，尾涡可能被人为削弱。

> 该修正方程是光滑解、常系数、固定 Courant 数条件下的局部渐近解释，不应把它当成所有非线性 CFD 求解器的通用黏性系数。

## 7. 实验 B：为什么“中心差分 + 显式欧拉”不稳定？

设某个离散 Fourier 模态为

$$
q_j^n=G^n\mathrm{e}^{\mathrm{i}j\theta},
$$

把它代入中心差分 + 显式欧拉：

$$
G_{\mathrm{central}}=1-\mathrm{i}C\sin\theta,
$$

因此

$$
|G_{\mathrm{central}}|^2=1+C^2\sin^2\theta >1
$$

（只要 $C\ne0$ 且 $\sin\theta\ne0$）。这些模式会随时间放大，最终导致不稳定。

为了更明显地观察现象，我们在光滑低频波上加入微弱高频扰动。高频幅值虽然只有 $10^{-3}$，但中心差分会不断放大它。

In [ ]:
C_test = 0.65
n_test = 60
q_noise = jnp.sin(2 * jnp.pi * x / L) + 0.001 * jnp.sin(2 * jnp.pi * (N // 4) * x / L)
q_center_bad = evolve_1d(q_noise, C_test, n_test, "central")
q_upwind_test = evolve_1d(q_noise, C_test, n_test, "upwind")
t_test = n_test * (C_test * dx / c)
q_noise_exact = jnp.sin(2*jnp.pi*(x - c*t_test)/L) + 0.001*jnp.sin(2*jnp.pi*(N//4)*(x-c*t_test)/L)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(np.asarray(x), np.asarray(q_noise_exact), 'k--', label='Exact')
axes[0].plot(np.asarray(x), np.asarray(q_upwind_test), label='Upwind', lw=1.3)
axes[0].set(title='Stable upwind', xlabel='x', ylabel='q')
axes[1].plot(np.asarray(x), np.asarray(q_noise_exact), 'k--', label='Exact')
axes[1].plot(np.asarray(x), np.asarray(q_center_bad), label='Central + Euler', lw=1.2)
axes[1].set(title='Unstable central + Euler', xlabel='x', ylabel='q')
for a in axes:
    a.legend()
    a.grid(alpha=0.25)
plt.tight_layout()
plt.show()
print('Central max |q|:', float(jnp.max(jnp.abs(q_center_bad))))
assert float(jnp.max(jnp.abs(q_center_bad))) > 2.0

### Von Neumann 放大因子示意

不同格式对一个 Fourier 模态的放大因子 $G$ 不同：

$$
\begin{aligned}
G_{\mathrm{upwind}}&=1-C(1-\mathrm{e}^{-\mathrm{i}\theta}),\\
G_{\mathrm{central}}&=1-\mathrm{i}C\sin\theta,\\
G_{\mathrm{LW}}&=1-\mathrm{i}C\sin\theta+C^2(\cos\theta-1).
\end{aligned}
$$

当 $|G|>1$ 时，该模态随时间增长。下面的图以 $C=0.65$ 为例，观察三种方法的放大因子模长。

In [ ]:
theta = np.linspace(0, np.pi, 400)
C_plot = 0.65
G_up = 1 - C_plot * (1 - np.exp(-1j*theta))
G_center = 1 - 1j*C_plot*np.sin(theta)
G_lw = 1 - 1j*C_plot*np.sin(theta) + C_plot**2*(np.cos(theta)-1)
fig, ax = plt.subplots(figsize=(8.5, 3.6))
ax.plot(theta, abs(G_up), label='Upwind + Euler')
ax.plot(theta, abs(G_center), label='Central + Euler')
ax.plot(theta, abs(G_lw), label='Lax–Wendroff')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='Neutral |G|=1')
ax.set(xlabel=r'Wave number $\theta$', ylabel=r'$|G|$', title='Von Neumann amplification (C=0.65)')
ax.grid(alpha=0.25)
ax.legend()
plt.show()
assert np.max(np.abs(G_up)) <= 1 + 1e-12
assert np.max(np.abs(G_lw)) <= 1 + 1e-12
assert np.max(np.abs(G_center)) > 1

## 8. 实验 C：间断问题与数值振荡

现在把光滑高斯脉冲改为周期方波：

$$
q_0(x)=\begin{cases}
1, & |d_L(x,x_0)|<w/2,\\
0, & \text{其他}.
\end{cases}
$$

对于真实线性对流，方波只会平移，不会变钝或出现超出 $[0,1]$ 的峰值。

一阶迎风法通常更平滑（数值耗散），Lax–Wendroff 在锋面附近可能出现过冲/欠冲（数值色散）。这也是现代高分辨率格式、限制器、TVD、ENO/WENO 等方法要解决的重要问题。

In [ ]:
width = 0.22
q_square = jnp.where(jnp.abs(periodic_distance(x, x0, L)) < width/2, 1.0, 0.0)
T_square = 0.27
steps_square, dt_square, C_square = step_count_for_target(T_square, dx, c, 0.65)
up_square = evolve_1d(q_square, C_square, steps_square, 'upwind')
lw_square = evolve_1d(q_square, C_square, steps_square, 'lax_wendroff')
exact_square = jnp.where(jnp.abs(periodic_distance(x, x0+c*T_square, L)) < width/2, 1.0, 0.0)

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.plot(np.asarray(x), np.asarray(exact_square), 'k--', lw=2, label='Exact top-hat')
ax.plot(np.asarray(x), np.asarray(up_square), label='Upwind', lw=1.7)
ax.plot(np.asarray(x), np.asarray(lw_square), label='Lax–Wendroff', lw=1.4)
ax.set(xlabel='x', ylabel='q(x,T)', title='Discontinuous top-hat: diffusion vs dispersion', ylim=(-0.35, 1.35))
ax.legend()
ax.grid(alpha=0.25)
plt.show()
print(f"Upwind range : [{float(jnp.min(up_square)):.4f}, {float(jnp.max(up_square)):.4f}]")
print(f"LW range     : [{float(jnp.min(lw_square)):.4f}, {float(jnp.max(lw_square)):.4f}]")
assert float(jnp.min(up_square)) >= -1e-12
assert float(jnp.max(up_square)) <= 1 + 1e-12
assert float(jnp.min(lw_square)) < -1e-3 or float(jnp.max(lw_square)) > 1.001

## 9. 从一维扩展到二维：真正的 CFD 过渡

二维常速度标量对流方程：

$$
\boxed{
\frac{\partial q}{\partial t}
+c_x\frac{\partial q}{\partial x}
+c_y\frac{\partial q}{\partial y}=0.
}
$$

若 $c_x>0,c_y>0$，二维一阶迎风 + 显式欧拉为：

$$
q_{i,j}^{n+1}=q_{i,j}^n
-C_x(q_{i,j}^n-q_{i-1,j}^n)
-C_y(q_{i,j}^n-q_{i,j-1}^n).
$$

其中 $C_x=c_x\Delta t/\Delta x$，$C_y=c_y\Delta t/\Delta y$。

对于这套线性方法，稳定（也保持单调性）的条件是：

$$
\boxed{C_x+C_y\le1.}
$$

**注意：**本节只是二维被动标量对流；之后 Navier–Stokes 的速度有空间变化和非线性，还需要求解压力以及不可压缩条件。

In [ ]:
@jax.jit
def step_upwind_2d(q, Cx, Cy):
    # First array axis represents x; second represents y.
    left = jnp.roll(q, 1, axis=0)
    bottom = jnp.roll(q, 1, axis=1)
    return q - Cx * (q - left) - Cy * (q - bottom)

@partial(jax.jit, static_argnames=("nsteps",))
def evolve_2d(q_initial, Cx, Cy, nsteps):
    return lax.fori_loop(
        0, nsteps, lambda _, q: step_upwind_2d(q, Cx, Cy), q_initial
    )

N2 = 96
Lx = Ly = 1.0
dx2 = Lx / N2
dy2 = Ly / N2
x2 = jnp.linspace(0.0, Lx, N2, endpoint=False)
y2 = jnp.linspace(0.0, Ly, N2, endpoint=False)
X2, Y2 = jnp.meshgrid(x2, y2, indexing='ij')
cx, cy = 0.7, 0.4
T2 = 0.35
# 保守地选择 Cx+Cy <= 0.60
nsteps2 = math.ceil(T2 * (cx/dx2 + cy/dy2) / 0.60)
dt2 = T2 / nsteps2
Cx, Cy = cx*dt2/dx2, cy*dt2/dy2
assert Cx + Cy <= 0.60 + 1e-12

center_x, center_y = 0.30, 0.55
sig2 = 0.075
q2_0 = gaussian(X2, center_x, sig2, Lx) * gaussian(Y2, center_y, sig2, Ly)
q2_exact = (gaussian(X2, center_x + cx*T2, sig2, Lx)
            * gaussian(Y2, center_y + cy*T2, sig2, Ly))
q2_final = evolve_2d(q2_0, Cx, Cy, nsteps2)

fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.4), constrained_layout=True)
for ax, data, label in zip(axes,
                           (q2_0, q2_final, q2_exact),
                           ('Initial', '2D upwind (JAX)', 'Exact translated')):
    im = ax.imshow(np.asarray(data).T, origin='lower', extent=[0,1,0,1],
                   vmin=0, vmax=1, cmap='viridis', interpolation='nearest')
    ax.set(title=label, xlabel='x', ylabel='y', aspect='equal')
fig.colorbar(im, ax=axes, fraction=0.023, pad=0.02)
plt.savefig('JAX_CFD_Lesson_02_preview.png', dpi=160, bbox_inches='tight')
plt.show()

mass2_initial = float(jnp.sum(q2_0) * dx2 * dy2)
mass2_final = float(jnp.sum(q2_final) * dx2 * dy2)
err2 = float(jnp.sqrt(jnp.sum((q2_final - q2_exact)**2) * dx2 * dy2))
print(f"2D: Cx={Cx:.4f}, Cy={Cy:.4f}, Cx+Cy={Cx+Cy:.4f}")
print(f"2D: mass difference={mass2_final-mass2_initial:+.3e}; L2 error={err2:.6e}")
assert abs(mass2_final-mass2_initial) < 1e-10
assert np.isfinite(err2) and err2 > 0

## 10. 课程小结：为什么这些知识与你的圆柱绕流毕设有关？

二维不可压缩 Navier–Stokes 动量方程为：

$$
\frac{\partial\boldsymbol{u}}{\partial t}
+\underbrace{(\boldsymbol{u}\cdot\nabla)\boldsymbol{u}}_{\text{对流}}
=-\frac1\rho\nabla p
+\underbrace{\nu\nabla^2\boldsymbol{u}}_{\text{黏性扩散}}.
$$

- 第一课：数值模拟黏性扩散 $\nu\nabla^2\boldsymbol u$ 的数学算子
- 第二课：认识对流离散、CFL 限制、数值耗散与色散
- 接下来：将两类算子组合，处理**压力梯度 + 不可压缩约束**，并建立二维验证用算例
- 之后：引入圆柱边界、升阻力计算、吹吸控制与代理模型数据接口

特别注意：**数值耗散可能改变涡脱落、升阻力频谱和后续强化学习的奖励。**因此，要做基于 CFD 的控制研究，仅仅看到漂亮的涡街远远不够，必须做网格/时间步收敛检查，以及基准算例比较。

## 11. LaTeX 进阶小抄

| 你要写的内容 | LaTeX 源码 |
|---|---|
| $q_j^{n+1}$ | `q_j^{n+1}` |
| $\partial q/\partial x$ | `\partial q/\partial x` |
| $\frac{q_j-q_{j-1}}{\Delta x}$ | `\frac{q_j-q_{j-1}}{\Delta x}` |
| $\left\lvert G\right\rvert$ | `\left\lvert G\right\rvert` |
| $\mathrm{e}^{\mathrm{i}\theta}$ | `\mathrm{e}^{\mathrm{i}\theta}` |
| $\underbrace{\cdots}_{\text{数值耗散}}$ | `\underbrace{\cdots}_{\text{数值耗散}}` |
| $O(\Delta x^2)$ | `O(\Delta x^2)` |
| $\nabla\cdot\boldsymbol u$ | `\nabla\cdot\boldsymbol u` |

**三个常见规则：**

1. 上下标超过一个字符，用花括号：`q_{i,j}^{n+1}`。
2. `\mathrm{d}x`、`\mathrm{i}`、`\mathrm{e}` 可以将微分符号、虚数单位及自然常数写成直立体；变量通常用斜体。
3. 公式多行对齐用 `\begin{aligned}`、`&` 和 `\\`。中英文说明可以放在公式外，公式内短说明用 `\text{...}`。

**练习：**试着自己用 `aligned` 环境把一阶迎风与 Lax–Wendroff 写在同一组多行公式中。

## 12. 练习（建议按顺序完成）

**练习 A（基础）**：令 $C=0.2,0.5,0.9$，比较同一个终止时间下迎风结果的峰值、宽度和 $L^2$ 误差。解释数值耗散随 CFL 如何变化。

**练习 B（核心）**：让网格数 $N=64,128,256,512$，固定终止时间及目标 CFL，记录光滑初值下迎风格式与 Lax–Wendroff 的 $L^2$ 误差。画对数坐标图估算收敛阶数。注意 Lax–Wendroff 的时间与空间精度均为二阶。

**练习 C（稳定性）**：把迎风 Courant 数改成 $C=1.1$，在光滑波中加入微弱高频扰动；观察增长并解释为什么超出稳定区间。请用独立单元格实验，不要覆盖已经验证的代码。

**练习 D（离散方向）**：令 $c<0$，在一维求解器中根据流速符号切换到**右侧迎风**公式；用解析平移解验证。（提示：正负速度的差分方向不同。）

**练习 E（二维）**：在二维程序中分别改变 $c_x,c_y$，让脉冲穿越右边界和上边界，检查质量守恒与运动方向。

### 留给你的空白实验单元格

下面代码单元格不会自动修改求解器，请自己补充试验。

In [ ]:
# TODO 练习 A：用不同目标 CFL 重复一维试验，画在同一张图上。
# 提示：
# for target_cfl in [0.2, 0.5, 0.9]:
#     n_i, dt_i, C_i = step_count_for_target(T, dx, c, target_cfl)
#     q_i = evolve_1d(q0, C_i, n_i, 'upwind')
#     ...
print('练习单元格已准备好。请修改本单元格完成实验。')

## 13. 课程检查与下一课

完成本课前，你应该能不看答案写出：

$$
\begin{aligned}
\text{迎风:}\quad &q_j^{n+1}=q_j^n-C(q_j^n-q_{j-1}^n),\\
\text{Lax–Wendroff:}\quad &q_j^{n+1}=q_j^n-\frac C2(q_{j+1}^n-q_{j-1}^n)
+\frac{C^2}{2}(q_{j+1}^n-2q_j^n+q_{j-1}^n).
\end{aligned}
$$

同时能够解释：**守恒性、稳定性、精度、数值耗散、数值色散是不同的性质**，不能彼此替代。

下一课进入二维 Navier–Stokes 数值求解器准备：**离散梯度 / 散度算子、交错网格与压力投影法**。

### 延伸阅读（官方文档）

- [JAX Quickstart](https://docs.jax.dev/en/latest/quickstart.html)
- [JAX `jnp.roll`](https://docs.jax.dev/en/latest/_autosummary/jax.numpy.roll.html)
- [JAX `jax.jit`](https://docs.jax.dev/en/latest/jit-compilation.html)
- [JAX `lax.fori_loop`](https://docs.jax.dev/en/latest/_autosummary/jax.lax.fori_loop.html)